In [1]:
import numpy as np
import torch
from stable_baselines3 import DQN
from mental_env import MentalHealthEnv

env = MentalHealthEnv()
model = DQN.load("dqn_mental_health finall.zip", env=env)

print("🔹 UNIT TESTING STARTED")

# Test 1: Input range
state = np.array([0.3, 0.2, 0.4, 0.1, 0.3])
print("Input Range Test:", np.all(state >= 0) and np.all(state <= 1))

# Test 2: Prediction
state = state.astype(np.float32)
action, _ = model.predict(state)
print("Prediction Test:", action in [0,1,2,3])

# Test 3: Confidence
with torch.no_grad():
    q_vals = model.q_net(torch.tensor(state).unsqueeze(0)).cpu().numpy()[0]

confidence = max(q_vals)
print("Confidence Test:", confidence is not None)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🔹 UNIT TESTING STARTED
Input Range Test: True
Prediction Test: True
Confidence Test: True


In [2]:
print("\n🔹 INTEGRATION TESTING STARTED")

# Simulate GUI input
input_data = np.array([0.5, 0.4, 0.3, 0.2, 0.1], dtype=np.float32)

action, _ = model.predict(input_data)

risk_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

result = risk_map[int(action)]

print("Pipeline Test:", result in risk_map.values())
print("Predicted Risk:", result)


🔹 INTEGRATION TESTING STARTED
Pipeline Test: True
Predicted Risk: Moderate


In [3]:
print("\n🔹 VALIDATION TESTING STARTED")

# Low values
state_low = np.zeros(5, dtype=np.float32)
action_low, _ = model.predict(state_low)
print("Low Input Test:", action_low is not None)

# High values
state_high = np.ones(5, dtype=np.float32)
action_high, _ = model.predict(state_high)
print("High Input Test:", action_high is not None)

# Random tests
success = True
for _ in range(10):
    state = np.random.rand(5).astype(np.float32)
    action, _ = model.predict(state)
    if action is None:
        success = False

print("Random Input Test:", success)


🔹 VALIDATION TESTING STARTED
Low Input Test: True
High Input Test: True
Random Input Test: True


In [4]:
import time

print("\n🔹 PERFORMANCE EVALUATION STARTED")

num_tests = 50
correct = 0

start_time = time.time()

for _ in range(num_tests):
    state = np.random.rand(5).astype(np.float32)
    action, _ = model.predict(state)

    if action in [0,1,2,3]:
        correct += 1

end_time = time.time()

accuracy = (correct / num_tests) * 100
avg_time = (end_time - start_time) / num_tests

print("Accuracy:", accuracy, "%")
print("Average Prediction Time:", avg_time, "seconds")


🔹 PERFORMANCE EVALUATION STARTED
Accuracy: 100.0 %
Average Prediction Time: 0.000736689567565918 seconds


In [5]:
print("🔹 UNIT TESTING OUTPUT")

state = np.array([0.3, 0.2, 0.4, 0.1, 0.3], dtype=np.float32)

# Prediction
action, _ = model.predict(state)

# Q-values
with torch.no_grad():
    q_vals = model.q_net(torch.tensor(state).unsqueeze(0)).cpu().numpy()[0]

print("Input State:", state)
print("Predicted Action:", action)
print("Q-values:", q_vals)
print("Max Q-value (Confidence Base):", np.max(q_vals))

🔹 UNIT TESTING OUTPUT
Input State: [0.3 0.2 0.4 0.1 0.3]
Predicted Action: 0
Q-values: [28.231066 27.491032 26.596102 26.53613 ]
Max Q-value (Confidence Base): 28.231066


In [6]:
print("\n🔹 INTEGRATION TESTING OUTPUT")

input_data = np.array([0.5, 0.4, 0.3, 0.2, 0.1], dtype=np.float32)

action, _ = model.predict(input_data)

risk_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

label = risk_map[int(action)]

print("Input Data:", input_data)
print("Predicted Action:", action)
print("Risk Level:", label)


🔹 INTEGRATION TESTING OUTPUT
Input Data: [0.5 0.4 0.3 0.2 0.1]
Predicted Action: 1
Risk Level: Moderate


In [7]:
print("\n🔹 VALIDATION TESTING OUTPUT")

# Low values
low = np.zeros(5, dtype=np.float32)
action_low, _ = model.predict(low)

# High values
high = np.ones(5, dtype=np.float32)
action_high, _ = model.predict(high)

print("Low Input:", low, "→ Prediction:", action_low)
print("High Input:", high, "→ Prediction:", action_high)

# Random samples
for i in range(3):
    state = np.random.rand(5).astype(np.float32)
    action, _ = model.predict(state)
    print(f"Random Test {i+1}:", state, "→", action)


🔹 VALIDATION TESTING OUTPUT
Low Input: [0. 0. 0. 0. 0.] → Prediction: 0
High Input: [1. 1. 1. 1. 1.] → Prediction: 3
Random Test 1: [0.72456586 0.9117344  0.88490397 0.8014968  0.6516267 ] → 3
Random Test 2: [0.34256792 0.43048912 0.11434294 0.24357924 0.08776534] → 0
Random Test 3: [0.4278205  0.965165   0.4147624  0.39819196 0.02009849] → 1


In [8]:
import time

print("\n🔹 PERFORMANCE EVALUATION OUTPUT")

num_tests = 50
times = []

start = time.time()

for _ in range(num_tests):
    state = np.random.rand(5).astype(np.float32)

    t1 = time.time()
    action, _ = model.predict(state)
    t2 = time.time()

    times.append(t2 - t1)

end = time.time()

print("Total Tests:", num_tests)
print("Average Time per Prediction:", np.mean(times), "seconds")
print("Min Time:", np.min(times))
print("Max Time:", np.max(times))
print("Total Execution Time:", end - start)


🔹 PERFORMANCE EVALUATION OUTPUT
Total Tests: 50
Average Time per Prediction: 0.0001532125473022461 seconds
Min Time: 0.0
Max Time: 0.0015687942504882812
Total Execution Time: 0.008667230606079102
